In [0]:
from pyspark.sql import functions as F

transactions = spark.table(
    "workspace.pyspark_deep_dive.transactions"
)

customers = spark.table(
    "workspace.pyspark_deep_dive.customers"
)

products = spark.table(
    "workspace.pyspark_deep_dive.products"
)

print("Transactions:", transactions.count())
print("Customers:", customers.count())
print("Products:", products.count())

In [0]:
transactions.printSchema()

In [0]:
transactions.filter(
    F.col("transaction_id").isNull()
).count()

In [0]:
transactions.filter(F.col("quantity").isNull()).count()

In [0]:
transactions.filter(F.col("quantity")<=0).count()

In [0]:
transactions.groupBy(F.col("transaction_id"))\
    .count()\
        .filter(F.col("count")>1).show()


In [0]:
quality_report = transactions.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_id"),
    F.sum(F.col("product_id").isNull().cast("int")).alias("null_product_id"),
    F.sum(F.col("quantity").isNull().cast("int")).alias("null_quantity"),
    F.sum((F.col("quantity") <= 0).cast("int")).alias("invalid_quantity"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("invalid_price"),
    F.sum((F.col("customer_id") == 0).cast("int")).alias("invalid_customer_id")
)

display(quality_report)

In [0]:
duplicate_transactions = (
    transactions
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_transactions)

In [0]:
valid_transactions = (
    transactions
    .join(
        customers,
        transactions.customer_id == customers.customer_id,
        "inner"
    )
)

invalid_transactions = (
    transactions
    .join(
        customers,
        transactions.customer_id == customers.customer_id,
        "left_anti"
    )
)

In [0]:
print("Valid:", valid_transactions.count())
print("Rejected:", invalid_transactions.count())

In [0]:
enriched_transactions = (
    transactions.alias("t")
    .join(
        customers.alias("c"),
        F.col("t.customer_id") == F.col("c.customer_id"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("t.product_id") == F.col("p.product_id"),
        "inner"
    )
    .select(
        F.col("t.transaction_id"),
        F.col("t.customer_id"),
        F.col("c.customer_name"),
        F.col("c.city"),
        F.col("c.customer_segment"),
        F.col("t.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("t.quantity"),
        F.col("t.unit_price"),
        F.round(
            F.col("t.quantity") * F.col("t.unit_price"), 2
        ).alias("total_amount"),
        F.col("t.transaction_date")
    )
)

display(enriched_transactions.limit(20))

In [0]:
print("Rows:", enriched_transactions.count())

In [0]:
invalid_products = (
    transactions
    .join(
        products,
        transactions.product_id == products.product_id,
        "left_anti"
    )
)

print("Transactions with invalid product_id:", invalid_products.count())

In [0]:
enriched_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.pyspark_deep_dive.enriched_transactions"
    )

In [0]:
enriched_transactions = spark.table(
    "workspace.pyspark_deep_dive.enriched_transactions"
)

In [0]:
print(enriched_transactions.count())